# Phase 1: Skill Normalization Demo

This notebook demonstrates the 3-tier skill normalization pipeline:
- **Tier 1**: Rule-based normalization (lowercase, versions, abbreviations, typos)
- **Tier 2**: Embedding-based semantic clustering
- **Tier 3**: Co-occurrence graph disambiguation


In [ ]:
import sys
sys.path.append('../src')

import pandas as pd
import numpy as np
from collections import Counter

from candidate_clustering.skills.normalizer import SkillNormalizer
from candidate_clustering.data.sample_generator import SampleDataGenerator

## 1. Generate Sample Data

Let's create synthetic candidate data with noisy skill names.

In [ ]:
# Generate 200 candidates
df = SampleDataGenerator.generate_candidates(n_candidates=200, seed=42)

print(f"Generated {len(df)} candidates")
print(f"\nProfile types: {df['profile_type'].value_counts().to_dict()}")

# Display first few candidates
df.head()

In [ ]:
# Examine raw skills (before normalization)
all_raw_skills = [skill for skills_list in df['skills'] for skill in skills_list]
print(f"Total raw skills (with duplicates): {len(all_raw_skills)}")
print(f"Unique raw skills: {len(set(all_raw_skills))}")

print("\nTop 20 most common raw skills:")
Counter(all_raw_skills).most_common(20)

## 2. Tier 1: Rule-Based Normalization

Apply rule-based normalization (lowercase, remove versions, expand abbreviations, fix typos)

In [ ]:
# Initialize normalizer
normalizer = SkillNormalizer()

# Test Tier 1 on some examples
test_skills = [
    "Python3.9",
    "JavaScript",
    "K8s",
    "Docker Container",
    "React.js",
    "pyton",
    "ML",
    "postgre"
]

print("Tier 1 Normalization Examples:")
print("-" * 50)
for skill in test_skills:
    normalized = normalizer.normalize_tier1(skill)
    print(f"{skill:20} → {normalized}")

In [ ]:
# Apply Tier 1 to all skills in dataset
unique_raw_skills = list(set(all_raw_skills))
tier1_mappings = {skill: normalizer.normalize_tier1(skill) for skill in unique_raw_skills}

print(f"After Tier 1:")
print(f"  Unique raw skills: {len(unique_raw_skills)}")
print(f"  Unique normalized skills: {len(set(tier1_mappings.values()))}")
print(f"  Reduction: {len(unique_raw_skills) - len(set(tier1_mappings.values()))} skills")

## 3. Tier 2: Embedding-Based Semantic Clustering

Group semantically similar skills using sentence embeddings and HDBSCAN clustering.

In [ ]:
# Get unique tier1-normalized skills
tier1_skills = list(set(tier1_mappings.values()))

print(f"Processing {len(tier1_skills)} unique skills with embeddings...")
print("This may take a minute...\n")

# Apply Tier 2
tier2_mappings = normalizer.normalize_tier2(tier1_skills)

print(f"\nAfter Tier 2:")
print(f"  Input skills: {len(tier1_skills)}")
canonical_skills = set([canonical for canonical, conf in tier2_mappings.values()])
print(f"  Unique canonical skills: {len(canonical_skills)}")
print(f"  Further reduction: {len(tier1_skills) - len(canonical_skills)} skills")

In [ ]:
# Show some clustering examples
print("\nTier 2 Clustering Examples:")
print("-" * 70)

# Group by canonical skill
from collections import defaultdict
clusters = defaultdict(list)
for skill, (canonical, conf) in tier2_mappings.items():
    clusters[canonical].append((skill, conf))

# Show clusters with multiple members
for canonical, members in sorted(clusters.items()):
    if len(members) > 1:
        print(f"\nCanonical: {canonical}")
        for skill, conf in members:
            print(f"  ← {skill:30} (confidence: {conf:.3f})")

## 4. Tier 3: Co-occurrence Graph Disambiguation

Use skill co-occurrence patterns to disambiguate ambiguous skills based on context.

In [ ]:
# Build co-occurrence graph from candidate data
candidate_skills_lists = df['skills'].tolist()

normalizer.build_cooccurrence_graph(
    candidate_skills_lists,
    min_cooccurrence=3
)

print("\nCo-occurrence Graph Statistics:")
stats = normalizer.get_statistics()
for key, value in stats.items():
    print(f"  {key}: {value}")

## 5. End-to-End Batch Normalization

Normalize all skills in the dataset using the complete pipeline.

In [ ]:
# Apply full normalization pipeline
final_mappings = normalizer.normalize_batch(unique_raw_skills, use_tier2=True)

print("\nFinal Normalization Results:")
print(f"  Original unique skills: {len(unique_raw_skills)}")
final_canonical = set([canonical for canonical, conf in final_mappings.values()])
print(f"  Final canonical skills: {len(final_canonical)}")
print(f"  Total reduction: {len(unique_raw_skills) - len(final_canonical)} skills")
print(f"  Compression ratio: {len(final_canonical) / len(unique_raw_skills):.1%}")

In [ ]:
# Show mapping examples
print("\nSample Mappings (Raw → Canonical):")
print("-" * 70)

sample_mappings = list(final_mappings.items())[:30]
for raw_skill, (canonical, confidence) in sorted(sample_mappings):
    if raw_skill != canonical:
        print(f"{raw_skill:25} → {canonical:25} (conf: {confidence:.2f})")

## 6. Apply Normalization to Dataset

Add normalized skills to our candidate dataset.

In [ ]:
# Apply normalization to each candidate
def normalize_candidate_skills(skills_list):
    normalized = []
    for skill in skills_list:
        canonical, confidence = final_mappings.get(skill, (skill, 1.0))
        normalized.append(canonical)
    return list(set(normalized))  # Remove duplicates

df['normalized_skills'] = df['skills'].apply(normalize_candidate_skills)

# Show before/after for some candidates
print("\nCandidate Skill Normalization Examples:")
print("=" * 80)

for idx in range(5):
    print(f"\nCandidate {df.iloc[idx]['candidate_id']} ({df.iloc[idx]['profile_type']}):")
    print(f"  Raw skills ({len(df.iloc[idx]['skills'])}): {df.iloc[idx]['skills']}")
    print(f"  Normalized ({len(df.iloc[idx]['normalized_skills'])}): {df.iloc[idx]['normalized_skills']}")

## 7. Save Mappings and Normalized Data

In [ ]:
# Save skill mappings
normalizer.save_mappings('../data/processed/skill_mappings.json')

# Save normalized candidate data
df.to_pickle('../data/processed/candidates_normalized.pkl')

print("✓ Saved skill mappings to: data/processed/skill_mappings.json")
print("✓ Saved normalized candidates to: data/processed/candidates_normalized.pkl")

## 8. Validation & Quality Metrics

In [ ]:
# Calculate quality metrics
print("\nNormalization Quality Metrics:")
print("=" * 70)

# 1. Compression ratio
compression = len(final_canonical) / len(unique_raw_skills)
print(f"Compression Ratio: {compression:.1%} ({len(final_canonical)}/{len(unique_raw_skills)} skills)")

# 2. Average confidence
avg_confidence = np.mean([conf for _, conf in final_mappings.values()])
print(f"Average Confidence: {avg_confidence:.3f}")

# 3. Low confidence mappings (may need manual review)
low_conf = [(skill, canonical, conf) 
            for skill, (canonical, conf) in final_mappings.items() 
            if conf < 0.85]
print(f"Low Confidence Mappings (<0.85): {len(low_conf)}")

if low_conf:
    print("\nLow Confidence Examples (for manual review):")
    for skill, canonical, conf in low_conf[:10]:
        print(f"  {skill:25} → {canonical:25} (conf: {conf:.2f})")

# 4. Most common canonical skills
all_normalized = [skill for skills_list in df['normalized_skills'] for skill in skills_list]
print(f"\nTop 15 Most Common Canonical Skills:")
for skill, count in Counter(all_normalized).most_common(15):
    print(f"  {skill:25} : {count:3d} occurrences")

## Summary

Phase 1 Implementation Complete!

✅ **Tier 1**: Rule-based normalization (lowercase, versions, abbreviations, typos)  
✅ **Tier 2**: Embedding-based semantic clustering  
✅ **Tier 3**: Co-occurrence graph disambiguation  

**Next Steps**: Phase 2 - Feature Engineering